In [3]:
!uv --version

uv 0.11.19 (7b2cff1c3 2026-06-03 x86_64-pc-windows-msvc)


In [4]:
!uv sync

Resolved 180 packages in 2ms
Checked 157 packages in 388ms


# Crossref

In [1]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
sys.path.append(str(PROJECT_DIR / "src"))

In [4]:
from dataclasses import asdict, dataclass
from importlib.resources import path
from pathlib import Path
from urllib import response

from core.config import Settings
from src.core.utils import write_json


@dataclass(frozen=True)
class PaperRecord:
    paper_id: str
    title: str
    summary: str
    authors: list[str]
    categories: list[str]
    primary_category: str
    published: str
    updated: str
    abs_url: str
    pdf_url: str
    comment: str


def parse_crossref_payload(payload: dict) -> list[PaperRecord]:
    """TODO(student): parse Crossref payload thanh list PaperRecord.

    Pseudo-code:
    1. Duyet `payload["message"]["items"]`.
    2. Lay DOI, title, abstract, authors, subject, dates, URLs.
    3. Chuan hoa text va bo record khong hop le.
    4. Tra ve list `PaperRecord`.
    """
    #raise NotImplementedError("Student task: implement Crossref payload parsing.")
    records = []
    for item in payload["message"]["items"]:
        # Extract relevant information from each item
        paper_id = item.get("DOI", "")
        title = item.get("title", [""])[0]
        summary = item.get("abstract", "")
        authors = [author.get("name", "") for author in item.get("author", [])]
        categories = item.get("subject", [])
        primary_category = item.get("primary_category", "")
        published = item.get("published", "")
        updated = item.get("updated", "")
        abs_url = item.get("URL", "")
        pdf_url = item.get("PDF", "")
        comment = item.get("comment", "")

        # Create a PaperRecord instance
        record = PaperRecord(
            paper_id=paper_id,
            title=title,
            summary=summary,
            authors=authors,
            categories=categories,
            primary_category=primary_category,
            published=published,
            updated=updated,
            abs_url=abs_url,
            pdf_url=pdf_url,
            comment=comment
        )
        records.append(record)
    return records

def fetch_source_records(settings: Settings) -> list[PaperRecord]:
    """TODO(student): goi source API, luu raw response, parse thanh records.

    Pseudo-code:
    1. Tao params tu `settings.source_query`, `settings.source_filter`, `settings.max_results`.
    2. Goi API voi retry cho cac status code nhu 429/503.
    3. Luu raw response vao `settings.paths.raw_api_response`.
    4. Parse payload bang `parse_crossref_payload`.
    5. Luu records vao `settings.paths.raw_records_json`.
    """
    #raise NotImplementedError("Student task: implement source fetching.")
    import requests
    import time
    MAX_RETRIES = 3
    RETRY_STATUS_CODES = {429, 503}
    params = {
        "query": settings.source_query,
        "filter": settings.source_filter,
        "rows": settings.max_results,
        "select": "DOI,title,abstract,author,published,published-online,published-print,issued,URL,subject,type,indexed,resource,container-title",
        "mailto": "student-lab@example.com",
    }

    headers = {
    "User-Agent": "day10-data-observability-lab/0.1 (mailto:student-lab@example.com)"
    }
    response = None
    for attempt in range(MAX_RETRIES + 1):
        response = requests.get(
            settings.source_api,
            params=params,
            headers=headers,
            timeout=30,
        )
        print(response.url)
        if response.status_code not in RETRY_STATUS_CODES:
            break

        if attempt == MAX_RETRIES:
            break
        
        retry_after = response.headers.get("Retry-After")
        if retry_after and retry_after.isdigit():
            wait_seconds = int(retry_after)
        else:
            wait_seconds = 2 ** attempt

        time.sleep(wait_seconds)
    response.raise_for_status()
    
    payload = response.json()
    write_json(settings.paths.raw_api_response, payload)
    
    records = parse_crossref_payload(payload)
    write_json(settings.paths.raw_records_json, [asdict(record) for record in records])
    
    return records
    

def load_raw_records(path: Path) -> list[PaperRecord]:
    """TODO(student): doc JSON snapshot va map thanh `PaperRecord`."""
    #raise NotImplementedError("Student task: implement raw record loading.")
    import json
    with open(path, "r") as f:
        data = json.load(f)
    return [PaperRecord(**record) for record in data]

In [5]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
sys.path.append(str(PROJECT_DIR / "src"))

from core.config import load_settings

settings = load_settings(PROJECT_DIR)

records = fetch_source_records(settings)

print(len(records))
print(records[0])

https://api.crossref.org/works?query=agentic+retrieval+augmented+generation+large+language+model&filter=from-pub-date%3A2025-12-12%2Chas-abstract%3Atrue&rows=24&select=DOI%2Ctitle%2Cabstract%2Cauthor%2Cpublished%2Cpublished-online%2Cpublished-print%2Cissued%2CURL%2Csubject%2Ctype%2Cindexed%2Cresource%2Ccontainer-title&mailto=student-lab%40example.com
24
PaperRecord(paper_id='10.6028/nist.ir.8579a', title='Mitigating Threats to Large Language Model and Retrieval-Augmented Generation Systems', summary='<jats:p/>', authors=[''], categories=[], primary_category='', published={'date-parts': [[2026]]}, updated='', abs_url='https://doi.org/10.6028/nist.ir.8579a', pdf_url='', comment='')


# Corruption

In [ ]:
import pandas as pd
df = pd.DataFrame([asdict(record) for record in records])

'<jats:p>This study investigates a method that integrates retrieval-augmented mechanisms into large language model agents for scientific literature review generation. The approach addresses the limitations of traditional review models that rely on parametric knowledge with insufficient timeliness and limited coverage. Incorporating external document retrieval and dynamic information fusion into the generation process enhances the accuracy and completeness of the output. The overall framework consists of query encoding, semantic retrieval, document filtering, knowledge fusion, language modeling, task planning, memory storage, and reinforcement optimization, forming a closed loop of retrieval, understanding, and generation. Relevant document fragments are first retrieved through semantic vector search to ensure comprehensive and reliable information sources. These external representations are then integrated with the internal embeddings of the language model through weighted fusion, whic

In [46]:
from datetime import date

def parse_crossref_date(value):
    if not isinstance(value, dict):
        return None

    date_parts = value.get("date-parts", [[None]])
    if not date_parts or not date_parts[0]:
        return None

    parts = date_parts[0]

    year = parts[0] if len(parts) > 0 else None
    month = parts[1] if len(parts) > 1 else 1
    day = parts[2] if len(parts) > 2 else 1

    if year is None:
        return None

    return date(year, month, day)

In [26]:
import re

# Function to remove HTML Tags
def normalize_text(text):
    pattern = re.compile('<.*?>')
    text = text.replace("\n", " ").replace("\r", " ")
    return pattern.sub(r'', text)

In [49]:
import re
import html

def clean_summary(value):
    if not isinstance(value, str):
        return ""

    text = html.unescape(value)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [6]:
from datetime import datetime

import pandas as pd

from ingestion.crossref import PaperRecord


def build_clean_dataframe(records: list[PaperRecord], run_date: datetime) -> pd.DataFrame:
    """TODO(student): clean raw records thanh dataframe san sang de embed.

    Pseudo-code:
    1. Normalize title, summary, authors, categories.
    2. Parse published/updated date.
    3. Tinh age_days.
    4. Tao cot helper:
       - authors_joined
       - categories_joined
       - summary_chars
       - text_for_embedding
    5. Drop duplicates va filter row xau.
    6. Sort dataframe va return.
    """
    #raise NotImplementedError("Student task: implement cleaning pipeline.")
    df = pd.DataFrame([asdict(record) for record in records])
    df["published"] = pd.to_datetime(df["published"], errors="coerce")
    df["updated"] = pd.to_datetime(df["updated"], errors="coerce")


In [ ]:
from pathlib import Path
from typing import Any
import json
import pandas as pd


def build_test_set(df: pd.DataFrame, output_path) -> list[dict[str, Any]]:
    """Tao bo evaluation set tu cleaned dataframe."""
    required_cols = {"paper_id", "title", "summary"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"Dataframe is missing required columns: {sorted(missing_cols)}")

    clean_df = df.copy()
    clean_df = clean_df.dropna(subset=["paper_id", "title", "summary"])
    clean_df = clean_df[
        (clean_df["paper_id"].astype(str).str.strip() != "")
        & (clean_df["title"].astype(str).str.strip() != "")
        & (clean_df["summary"].astype(str).str.strip() != "")
    ]

    if len(clean_df) < 3:
        raise ValueError("Need at least 3 valid documents to build a test set.")

    if "published" in clean_df.columns:
        clean_df = clean_df.sort_values("published", ascending=False, na_position="last")

    selected = clean_df.head(min(6, len(clean_df)))
    test_set = []

    def add_case(question_type, question, ground_truth, paper_id):
        if ground_truth is None:
            return

        ground_truth = str(ground_truth).strip()
        if not ground_truth:
            return

        test_set.append(
            {
                "id": f"q{len(test_set) + 1:03d}",
                "question_type": question_type,
                "question": question,
                "ground_truth": ground_truth,
                "ground_truth_doc_ids": [str(paper_id)],
            }
        )

    for _, row in selected.iterrows():
        paper_id = row["paper_id"]
        title = str(row["title"]).strip()

        add_case(
            "summary",
            f"What is the main idea of the paper titled '{title}'?",
            row.get("summary", ""),
            paper_id,
        )

        authors = row.get("authors_joined", "")
        if not authors and "authors" in row:
            raw_authors = row.get("authors")
            if isinstance(raw_authors, list):
                authors = ", ".join(str(author).strip() for author in raw_authors if str(author).strip())

        add_case(
            "authors",
            f"Who are the authors of the paper titled '{title}'?",
            authors,
            paper_id,
        )

        add_case(
            "date",
            f"When was the paper titled '{title}' published?",
            row.get("published", ""),
            paper_id,
        )

        categories = row.get("categories_joined", "")
        if not categories and "categories" in row:
            raw_categories = row.get("categories")
            if isinstance(raw_categories, list):
                categories = ", ".join(
                    str(category).strip() for category in raw_categories if str(category).strip()
                )

        add_case(
            "categories",
            f"What categories or subjects are associated with the paper titled '{title}'?",
            categories,
            paper_id,
        )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(test_set, indent=2, ensure_ascii=True) + "\n", encoding="utf-8")

    return test_set